<a href="https://colab.research.google.com/github/muktiranisutradhar1-debug/special-parakeet/blob/main/Weather_forecasting_AUS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Install Libraries

In [ ]:
!pip install xgboost

#Import Libraries

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from sklearn.impute import SimpleImputer

from sklearn.ensemble import RandomForestClassifier

from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score
)

#Load Dataset

In [ ]:
df = pd.read_csv('/content/weatherAUS.csv')

df.head()

#Data Overview

In [ ]:
print(df.shape)

print(df.info())

print(df.isnull().sum())

#Handle Missing Values

In [ ]:
num_cols = df.select_dtypes(include=['float64','int64']).columns

cat_cols = df.select_dtypes(include=['object']).columns

num_imputer = SimpleImputer(strategy='median')
cat_imputer = SimpleImputer(strategy='most_frequent')

df[num_cols] = num_imputer.fit_transform(df[num_cols])

df[cat_cols] = cat_imputer.fit_transform(df[cat_cols])

#Encode Categorical Variables

In [ ]:
le = LabelEncoder()

for col in cat_cols:
    df[col] = le.fit_transform(df[col])

#Features and Target

In [ ]:
X = df.drop('RainTomorrow', axis=1)

y = df['RainTomorrow']

#Train, Validation, Test Split

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42
)

print(X_train.shape)
print(X_val.shape)
print(X_test.shape)

#Random Forest Model

In [ ]:
rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

rf.fit(X_train,y_train)

rf_pred = rf.predict(X_test)

#Evaluation

In [ ]:
print("Random Forest Accuracy:",
      accuracy_score(y_test, rf_pred))

print(classification_report(y_test, rf_pred))

print(confusion_matrix(y_test, rf_pred))

#XGBoost Model

In [ ]:
xgb = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    random_state=42
)

xgb.fit(X_train,y_train)

xgb_pred = xgb.predict(X_test)

#Evaluation

In [ ]:
print("XGBoost Accuracy:",
      accuracy_score(y_test,xgb_pred))

print(classification_report(y_test,xgb_pred))

print(confusion_matrix(y_test,xgb_pred))

#Compare Models

In [ ]:
rf_acc = accuracy_score(y_test, rf_pred)

xgb_acc = accuracy_score(y_test, xgb_pred)

print("Random Forest:", rf_acc)
print("XGBoost:", xgb_acc)

#ROC-AUC

In [ ]:
rf_auc = roc_auc_score(
    y_test,
    rf.predict_proba(X_test)[:,1]
)

xgb_auc = roc_auc_score(
    y_test,
    xgb.predict_proba(X_test)[:,1]
)

print("RF AUC =", rf_auc)

print("XGB AUC =", xgb_auc)